# Maize APAR / SPM — Thoothukudi & Siddipet (rabi 2022-23)

Earth Engine APAR and stress scalars, and SPM yield (RUE 4.65, HI 0.26, Tmax 34 °C) exported per grid cell. Also exports Sentinel-2 band zonal means at the Siddipet CCE points, for `exploratory/eda`.

**Inputs:** Thoothukudi grid, Siddipet CCE shapefile  
**Outputs:** `splits/apar/*.tif`, `Sentinel_cce.csv`  

> Update the path variables in the first cells before running. See [`docs/`](../../docs/) for methodology and parameters.

In [ ]:
import geemap
import ee
from geeS2downloader import GEES2Downloader
import os
import numpy as np

In [ ]:
from datetime import datetime

In [ ]:
ee.Authenticate()
ee.Initialize()

In [ ]:
gp_data=geemap.shp_to_ee(r"C:\Users\pushkargaur\Desktop\miscellaneous\Neha\YieldData\raw\Thuthookudi_Maize_Rabi2022-23\shape\Thuthukodi_grid.shp")
boundary = geemap.shp_to_ee(r"C:\Users\pushkargaur\Desktop\miscellaneous\Neha\YieldData\raw\Thuthookudi_Maize_Rabi2022-23\shape\Thuthukodi_dist.shp")
# cropmap = r"C:\Users\pushkargaur\Desktop\miscellaneous\Neha\YieldData\raw\Siddipet_Maize_Rabi2022-23\Siddipet_Maize_Rabi2022-23_gcs.tif"
cropmask = ee.Image('projects/ee-gaurpushkar8/assets/Thuthookudi_Maize_Rabi2022-23_gcs')
gp_bound=gp_data.getInfo()['features']

In [ ]:
cropmask=cropmask.gt(0)

In [ ]:
roi = boundary.first()
geom = roi.geometry()

In [ ]:
base=r'C:\Users\pushkargaur\Desktop\miscellaneous\Neha\YieldData\raw\Thuthookudi_Maize_Rabi2022-23\splits\apar'
if not os.path.exists(base):
    os.makedirs(base)

In [ ]:
# /// Growth stages


g0 = '2022-12-13'
g1 = '2023-01-20'
g2 = '2023-01-31'
g3 = '2023-03-23'
g4 = '2023-05-01'


In [ ]:
def typeCast(image):
    return image.toFloat()

In [ ]:
# // Temperature 

tmax = 34
tmin = 5
topt = 27

RUE = 4.65
HI = 0.26


In [ ]:
def temperaturestress(image):
    numerator = (image.subtract(ee.Image(tmin))).multiply(image.subtract(ee.Image(tmax))) 
    denominator = ((image.subtract(ee.Image(tmin))).multiply(image.subtract(ee.Image(tmax)))).subtract(
                  (image.subtract(ee.Image(topt))).multiply((image.subtract(ee.Image(topt)))))
    return typeCast(numerator.divide(denominator).rename('Tstress'))

In [ ]:
def local_max(ndwi):
    
    theMax = ndwi.reduceRegion({'reducer' : ee.Reducer.max(), 
                                'geometry' : boundary.geometry(), 
                                'maxPixels' : 74882817, 
                                'bestEffort' : True}
                               )
    return ndwi.set({'NDWI':theMax})

In [ ]:
# // water stress calculation

def waterscale(image):
    img = image.clip(boundary).multiply(0.0001)
    nir = image.select('sur_refl_b02')
    swir = image.select('sur_refl_b06')
    ndwi = nir.subtract(swir).divide(nir.add(swir)).rename('NDWI')
    gcs = (cropmask.projection())
    ndwi = ndwi.reproject(gcs)
    ndwi = ndwi.mask(cropmask)
#     theMax = ndwi.reduceRegion({'reducer' : ee.Reducer.max(), 
#                                 'geometry' : boundary.geometry(), 
# #                                 'maxPixels' : 74882817, 
#                                 'bestEffort' : True}
#                                )
#     theMax = local_max(ndwi)
    ws = ee.Image(1).subtract(ndwi).divide(ee.Image(1.39))
    ws = ws.reproject(gcs)
    return typeCast(ws)



In [ ]:
def fparscale(image):
    return typeCast(image).clip(boundary).multiply(0.01).mask(cropmask);

In [ ]:
def insolscale(image):
    return typeCast(image).clip(boundary).multiply(0.48).divide(1000000).mask(cropmask);

In [ ]:
def image_download_yield(image,filename,bound):
    image=image.multiply(1000)
    image = image.clip(bound)
    image = image.int16()
    if not os.path.exists(filename):
        print(filename)
        geemap.ee_export_image(image, filename=filename, scale=10, region=bound, file_per_band=False)

In [ ]:
def image_download(image,filename,bound):
    geemap.ee_export_image(image, filename=filename, scale=10, region=bound, file_per_band=False)

In [ ]:
def gp_image_download(image,date,factor):
    for d in np.arange(len(gp_bound)):
        fet = gp_bound[d]
        feature=ee.Feature(gp_data.filterMetadata('Id','equals',fet['properties']['Id']).first())
        fn=os.path.join(base,factor+'-'+date+'-'+str(d)+'.tif')
        image_download_yield(image,fn,feature.geometry())

In [ ]:
# // APAR calculation 
 
mod15a2h = ee.ImageCollection("MODIS/061/MOD15A2H").select('Fpar_500m').filter(ee.Filter.date(g0,g4)).filterBounds(boundary);
fpar = mod15a2h.map(fparscale);

apar_imgs = ee.List([]);

def parprocessing(date_i,date_e,date):
    global apar_imgs
    era5land = ee.ImageCollection("ECMWF/ERA5_LAND/DAILY_AGGR").select('surface_solar_radiation_downwards_sum').filter(ee.Filter.date(date_i,date_e)).filterBounds(boundary);
    par = era5land.map(insolscale);
    par_sum = par.sum().set('system:index',date);
    gcs = cropmask.projection();
    par_sum = par_sum.rename([date]);
    fpar_img = fpar.filterMetadata('system:index','equals',date).first().reproject(gcs);
    apar_img = typeCast(par_sum).multiply(typeCast(fpar_img)).rename([date]).set('system:index',date);
    apar_img = apar_img.clip(boundary)
    apar_img = apar_img.mask(cropmask);
    fn=os.path.join(base,'apar-',date+'.tif')
    gp_image_download(apar_img,date,'APAR')
    apar_imgs = apar_imgs.add(typeCast(apar_img));


fparinfo = (fpar.getInfo()['features'])
for d in np.arange(len(fparinfo)):
    date_i='-'.join(fparinfo[d]['properties']['system:index'].split('_'))
    print(date_i)
    if(d==len(fparinfo)-1):
        date_e=g4

    else:
        date_e='-'.join(fparinfo[d+1]['properties']['system:index'].split('_'))


    parprocessing(date_i,date_e,fparinfo[d]['properties']['system:index'])


apar = ee.ImageCollection(apar_imgs)


In [ ]:

mod09a1 = ee.ImageCollection('MODIS/061/MOD09A1').filter(ee.Filter.date(g2, g3)).filterBounds(boundary);
ndwi_coll = mod09a1.mean()
waterScaler = waterscale(ndwi_coll);
waterScaler = waterScaler.clip(boundary).mask(cropmask)
fn=os.path.join(base,'apar','waterscaler.tif')
gp_image_download(waterScaler,'2','WS')


era5_temp = ee.ImageCollection("ECMWF/ERA5_LAND/DAILY_AGGR").select('temperature_2m').filter(ee.Filter.date(g2, g3)).filterBounds(boundary);
era5_mean_temp = era5_temp.mean().subtract(ee.Image(273.15));
tstress = temperaturestress(era5_mean_temp);
tstress = tstress.clip(boundary).mask(cropmask)
fn=os.path.join(base,'apar','temp_stress.tif')
gp_image_download(tstress,'2','TS')


In [ ]:
# //// Growth stage calculations

# g0_date=ee.Date(g0)
# g1_date=ee.Date(g1)
# g2_date=ee.Date(g2)
# g3_date=ee.Date(g3)
# g4_date=ee.Date(g4)
g0_date = datetime.strptime(g0,'%Y-%m-%d')
g1_date = datetime.strptime(g1,'%Y-%m-%d')
g2_date = datetime.strptime(g2,'%Y-%m-%d')
g3_date = datetime.strptime(g3,'%Y-%m-%d')
g4_date = datetime.strptime(g4,'%Y-%m-%d')



aparinfo = (apar.getInfo()['features'])


g1_bin = ee.List([])
g2_bin = ee.List([])
g3_bin = ee.List([])
g4_bin = ee.List([])

def g1_bin_list(img):
    global g1_bin
    g1_bin = g1_bin.add(typeCast(img))


def g2_bin_list(img):
    global g2_bin
    g2_bin = g2_bin.add(typeCast(img))


def g3_bin_list(img,ts):
    global g3_bin
    img = img.multiply(ts)
    img=img.reproject(cropmask.projection())
    g3_bin = g3_bin.add(typeCast(img))

def g4_bin_list(img):
    global g4_bin
    g4_bin = g4_bin.add(typeCast(img))
    
for d in np.arange(len(aparinfo)):
    date = aparinfo[d]['properties']['system:index']
    img_date = datetime.strptime(date,'%Y_%m_%d')
    if img_date<g1_date:
        img = apar.filterMetadata('system:index','equals',date).first()
        g1_bin_list(img)
        
    elif ((img_date>g1_date) & (img_date <= g2_date)):
        img = apar.filterMetadata('system:index','equals',date).first()
        g2_bin_list(img)        
    
        
    elif ((img_date>g2_date) & (img_date <= g3_date)):
        img = apar.filterMetadata('system:index','equals',date).first()
        g3_bin_list(img,tstress)
    
    elif (img_date>g3_date):
        img = apar.filterMetadata('system:index','equals',date).first()
        g4_bin_list(img)     
        

g1_img_collection=ee.ImageCollection(g1_bin)

g2_img_collection=ee.ImageCollection(g2_bin)

g3_img_collection=ee.ImageCollection(g3_bin)

g4_img_collection=ee.ImageCollection(g4_bin)

gs1 = g1_img_collection.sum().rename(['G1'])
gs2 = g2_img_collection.sum().rename(['G2'])
gs3 = g3_img_collection.sum().rename(['G3'])
gs4 = g4_img_collection.sum().rename(['G4'])


yld = (gs1.add(gs2).add(gs3).add(gs4)).multiply(ee.Image(HI)).multiply(ee.Image(RUE))


In [ ]:
for d in np.arange(len(gp_bound)):
    fet = gp_bound[d]
    feature=ee.Feature(gp_data.filterMetadata('Id','equals',fet['properties']['Id']).first())
    fn=os.path.join(base,'Yield'+'_'+str(d)+'.tif')
#     image_download_yield(yld,fn,feature.geometry())

In [ ]:
cce_data=geemap.shp_to_ee(r"C:\Users\pushkargaur\Desktop\miscellaneous\Neha\YieldData\raw\Siddipet_Maize_Rabi2022-23\shape\Siddipet_cce.shp")

In [ ]:
s2_coll=ee.ImageCollection('COPERNICUS/S2_SR_HARMONIZED').filterBounds(cce_data).filterDate(g0,g4);
sentinel_data=s2_coll.select(['B2','B3','B4','B8','B11','B12'])

In [ ]:
tes=os.path.join(base,'Sentinel_cce.csv')
tes2=os.path.join(base,'Yield_cce.csv')

In [ ]:
geemap.zonal_statistics(
    sentinel_data, cce_data, tes, statistics_type='MEAN', scale=10, 
)